# Adaptive Query Execution (AQE)

Run this notebook cell-by-cell against the live cluster spawned from the topic page. See `concept.md` for the what/why background.

This topic deliberately toggles `spark.sql.adaptive.enabled` *within* the notebook (via `spark.conf.set`) so both the AQE-off and AQE-on cases are runnable in one session, regardless of what the cluster panel's AQE toggle was set to at spawn time (US-2.5's "runnable in both states" requirement).

In [1]:
import sys
sys.path.insert(0, "/workspace")

import random

from pyspark.sql import Row, SparkSession
from pyspark.sql import functions as F
from driver.playbook import checkpoint

spark = SparkSession.builder.appName("aqe").getOrCreate()
print("spark.sql.adaptive.enabled (cluster default):", spark.conf.get("spark.sql.adaptive.enabled"))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/14 18:27:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


spark.sql.adaptive.enabled (cluster default): true


## 1. Generate a deliberately skewed dataset inline

No `tools/datagen/` utility exists yet (a separate, un-scoped backlog item) -- this generates skew in-notebook instead, the same way `compose/smoke_test.py` and the partitioning/shuffle topic already self-generate data.

3 "hot" keys deliberately get ~60% of all rows; the remaining 40% spread evenly across 300 "normal" keys -- enough skew that the hot keys' shuffle partitions genuinely dominate the others' size.

In [2]:
FACT_ROWS = 4_000_000
NUM_NORMAL_KEYS = 300
HOT_KEYS = ["hot-0", "hot-1", "hot-2"]

def make_row(i: int) -> Row:
    # Deterministic ~60% hot-key skew: no numpy dependency, matches this
    # topic's inline-generation scope (tools/datagen/ is out of scope here).
    if i % 5 < 3:
        key = HOT_KEYS[i % len(HOT_KEYS)]
    else:
        key = f"key-{i % NUM_NORMAL_KEYS}"
    return Row(key=key, amount=float(i % 997))

fact_rdd = spark.sparkContext.parallelize(range(FACT_ROWS), 32).map(make_row)
fact_df = spark.createDataFrame(fact_rdd)

dim_df = spark.createDataFrame(
    [Row(key=k, label=f"label-{k}") for k in HOT_KEYS + [f"key-{i}" for i in range(NUM_NORMAL_KEYS)]]
)

# Confirm the skew actually landed (US-0.4-style verification): top keys' share of rows.
top_counts = fact_df.groupBy("key").count().orderBy(F.desc("count")).limit(5)
top_counts.show()

+------+------+
|   key| count|
+------+------+
| hot-0|800000|
| hot-2|800000|
| hot-1|800000|
|key-63| 13334|
|key-94| 13334|
+------+------+



## 2. Skewed join with AQE off -- hypothesize first

Disable broadcast (both sides are large enough that it wouldn't fire anyway, but pin it off for clarity) and disable AQE. Join `fact_df` to `dim_df` on the skewed `key`.

**Hypothesis:** do you expect task durations to be roughly even across the join's shuffle stage, or do you expect a small number of much slower tasks?

In [3]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.adaptive.enabled", "false")

skewed_join_no_aqe = fact_df.join(dim_df, on="key", how="inner")
# Materialize via foreach(), not count(): DataFrame.count() internally executes a *derived*
# groupBy().count() query, not this DataFrame's own plan object -- for a static (non-adaptive) plan
# that's harmless, but the AQE-on cells below need the SAME plan object actually executed so its
# adaptive state is real when we explain() it (verified against a real cluster while building this
# notebook: count() left the plan looking un-executed). foreach() with a no-op forces real execution
# without collecting the (large) result to the driver.
skewed_join_no_aqe.foreach(lambda row: None)
skewed_join_no_aqe.explain(mode="formatted")

== Physical Plan ==
* Project (10)
+- * SortMergeJoin Inner (9)
   :- * Sort (4)
   :  +- Exchange (3)
   :     +- * Filter (2)
   :        +- * Scan ExistingRDD (1)
   +- * Sort (8)
      +- Exchange (7)
         +- * Filter (6)
            +- * Scan ExistingRDD (5)


(1) Scan ExistingRDD [codegen id : 1]
Output [2]: [key#0, amount#1]
Arguments: [key#0, amount#1], MapPartitionsRDD[5] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter [codegen id : 1]
Input [2]: [key#0, amount#1]
Condition : isnotnull(key#0)

(3) Exchange
Input [2]: [key#0, amount#1]
Arguments: hashpartitioning(key#0, 200), ENSURE_REQUIREMENTS, [plan_id=79]

(4) Sort [codegen id : 2]
Input [2]: [key#0, amount#1]
Arguments: [key#0 ASC NULLS FIRST], false, 0

(5) Scan ExistingRDD [codegen id : 3]
Output [2]: [key#2, label#3]
Arguments: [key#2, label#3], MapPartitionsRDD[10] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitio

In [4]:
checkpoint(skewed_join_no_aqe, topic="aqe")
print("Checkpoint written -- Reveal self-check, then check the Stages tab / stage-metrics panel for "
      "per-task duration spread on the shuffle stage before moving on.")

Checkpoint written -- Reveal self-check, then check the Stages tab / stage-metrics panel for per-task duration spread on the shuffle stage before moving on.


## 3. Same join with AQE + skew-join handling on

Re-enable AQE and its skew-join split. Same query, same data -- only the adaptive execution settings change.

AQE's skew-join thresholds (`spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes`, default **256MB**, and `skewedPartitionFactor`, default 5x the median partition size) are tuned for production-scale shuffle partitions. This notebook's dataset is deliberately small enough to run quickly on a 3-worker/2-core/4GB cluster, so its skewed partitions -- while genuinely ~60x more *rows* than a normal key, confirmed above -- don't reach 256MB of *bytes*. Lower both thresholds so the skew AQE detects is proportional to this demo's scale, the same way you would tune them down for a smaller production workload rather than accept AQE silently not firing.

**Hypothesis:** what new node type do you expect to see in the plan, and where?

In [5]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "256k")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "2")
# Also tighten the coalescing target now (not just in step 4): the default 64MB advisory partition
# size lets coalescing merge partitions into a baseline large enough that the skewed partition no
# longer looks proportionally bigger (verified against a real cluster -- with only the skew thresholds
# lowered, the plan showed AQEShuffleRead(coalesced) but no skew split; adding this fixed it).
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "256k")

skewed_join_with_aqe = fact_df.join(dim_df, on="key", how="inner")
skewed_join_with_aqe.foreach(lambda row: None)  # see the foreach() note in step 2's cell above
skewed_join_with_aqe.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (22)
+- == Final Plan ==
   ResultQueryStage (15)
   +- * Project (14)
      +- * SortMergeJoin(skew=true) Inner (13)
         :- * Sort (6)
         :  +- AQEShuffleRead (5)
         :     +- ShuffleQueryStage (4), Statistics(sizeInBytes=122.1 MiB, rowCount=4.00E+6)
         :        +- Exchange (3)
         :           +- * Filter (2)
         :              +- * Scan ExistingRDD (1)
         +- * Sort (12)
            +- AQEShuffleRead (11)
               +- ShuffleQueryStage (10), Statistics(sizeInBytes=14.2 KiB, rowCount=303)
                  +- Exchange (9)
                     +- * Filter (8)
                        +- * Scan ExistingRDD (7)
+- == Initial Plan ==
   Project (21)
   +- SortMergeJoin Inner (20)
      :- Sort (17)
      :  +- Exchange (16)
      :     +- Filter (2)
      :        +- Scan ExistingRDD (1)
      +- Sort (19)
         +- Exchange (18)
            +- Filter (8)
               +- Scan ExistingRDD (7)


(1) Scan Exis

In [ ]:
checkpoint(skewed_join_with_aqe, topic="aqe")
print("Checkpoint written -- Reveal again and compare: look for the AQE adaptive-reader node, and "
      "compare this run's per-task duration spread on the Stages tab against step 2's.")

## 4. Post-shuffle partition coalescing

A `groupBy` after a filter that shrinks the data can leave `spark.sql.shuffle.partitions` (this cluster's default: 200) far larger than the post-filter data actually needs. With AQE on, Spark coalesces small adjacent post-shuffle partitions at runtime -- governed by `spark.sql.adaptive.advisoryPartitionSizeInBytes` (default 64MB: "try to make each post-shuffle partition roughly this big by merging small ones"). Same reasoning as step 3: lower it so coalescing has something to do at this notebook's deliberately small demo scale.

**Hypothesis:** do you expect the *executed* number of post-shuffle partitions to be smaller than 200?

In [ ]:
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "256k")

narrow_df = fact_df.filter(F.col("key").isin(HOT_KEYS[:1]))  # shrinks the data a lot before the shuffle

coalesced = narrow_df.groupBy("key").agg(F.count("*").alias("cnt"), F.avg("amount").alias("avg_amount"))
coalesced.foreach(lambda row: None)  # see the foreach() note in step 2's cell above
coalesced.explain(mode="formatted")

In [ ]:
checkpoint(coalesced, topic="aqe")
print("Checkpoint written -- Reveal again. Compare the initial spark.sql.shuffle.partitions=200 plan "
      "against the coalesced/executed partition count via the AQE adaptive-reader node and the "
      "stage-metrics panel's numTasks for this stage.")

## 5. Reset session-level confs


In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)